# packages

In [3]:
import os
from glob import glob
import re
import numpy as np
import cv2
from PIL import Image
import argparse
import shutil
import matplotlib.pyplot as plt
from utils import *

# code run only once

## video to frame

In [ ]:
# Call the function to convert the video to images
convert_video_to_images('src/drill/2.mp4', 'drill2')

Creating output folder: drill2
Total frames: 1144


## src video

1144 frames for src video, from no.360 to no.903 being drill bit.

every 24 frames for a round, 360-383, 384-407, 408-...

phase 1: 360-503 (included)

phase 2: 575-646 (included)

In [ ]:
# Create folders drill2-1 and drill2-2
os.makedirs("drill2-1", exist_ok=True)
os.makedirs("drill2-2", exist_ok=True)

# Define the ranges for extraction and renaming
ranges = [(360, 503, "drill2-1", 1, 144), (575, 646, "drill2-2", 1, 72)]

# Iterate over the ranges
for start, end, folder_name, new_start, _ in ranges:
    for i in range(start, end + 1):
        src = f"drill2/frame_{i}.jpg"
        dst = f"{folder_name}/image_{new_start}.jpg"
        
        if os.path.exists(src):
            shutil.copy(src, dst)
            new_start += 1

## functions

In [4]:
kernel=np.array((9,9), dtype=np.uint8)

In [5]:
def fd(src, new, output_name):
    # src: name of folder
    # new: name of folder
    src_paths = sorted(glob(f"{src}/*.jpg"), key=lambda x: int(re.search(r'(\d+).jpg', os.path.basename(x)).group(1)))
    new_paths = sorted(glob(f"{new}/*.jpg"), key=lambda x: int(re.search(r'(\d+).jpg', os.path.basename(x)).group(1)))
    
    thresh = 1000
    box_result = []
    for idx in range(0, len(src_paths)):
        # read frames
        frame1_bgr = cv2.imread(src_paths[idx])
        frame2_bgr = cv2.imread(new_paths[idx])

        # get detections
        detections = get_detections(cv2.cvtColor(frame1_bgr, cv2.COLOR_BGR2GRAY), 
                                    cv2.cvtColor(frame2_bgr, cv2.COLOR_BGR2GRAY), 
                                    bbox_thresh=thresh,
                                    nms_thresh=1e-4)
        box_result.append(detections)
    print('the box thresh: ', thresh)
    
    ####visualize
    if not os.path.exists('temp'):
        os.makedirs('temp')
    else:
        shutil.rmtree('temp')
        os.makedirs('temp')
        
    print('start to visualize the box on the images')

    for idx in range(0, len(src_paths)):
        # read frames
        frame_bgr = cv2.imread(new_paths[idx])
        detections = box_result[idx]                           
        # draw bounding boxes on frame
        draw_bboxes(frame_bgr, detections)

        # save image for GIF
        fig = plt.figure(figsize=(1280/100, 720/100)) # (15, 7)  / (1280/100, 720/100)
        plt.imshow(frame_bgr)
        plt.axis('off')
        fig.savefig(f"temp/frame_{idx}.png")
        plt.close()

    file_path = output_name

    # Check if the file exists before attempting to delete it
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

    create_gif_from_images(file_path, 'temp', '.png')

# code to execute

phase 1

In [6]:
# Define the ranges for extraction and renaming
ranges_2_1_src = [(1, 24, 'drill2-1', 1, 24)]
ranges_2_1 = [(25, 48, "drill2-1", 1, 'drill2-1-1.GIF'), 
          (49, 72, "drill2-1", 1, 'drill2-1-2.GIF'),
          (73, 96, "drill2-1", 1, 'drill2-1-3.GIF'),
          (97, 120, "drill2-1", 1, 'drill2-1-4.GIF'),
          (120, 144, "drill2-1", 1, 'drill2-1-5.GIF')]

ranges_2_2_src = [(1, 24, 'drill2-2', 1, 24)]
ranges_2_2 = [(25, 48, "drill2-2", 1, 'drill2-2-1.GIF'), 
          (49, 72, "drill2-2", 1, 'drill2-2-2.GIF')]

# Create folders
if os.path.exists('src_temp'):
    shutil.rmtree('src_temp')
os.makedirs("src_temp")

# Iterate over the ranges
for start, end, src_folder, new_start, _ in ranges_2_1_src:
    for i in range(start, end + 1):
        src = f"{src_folder}/image_{i}.jpg"
        dst = f"src_temp/image_{new_start}.jpg"
        
        if os.path.exists(src):
            shutil.copy(src, dst)
            new_start += 1

# Iterate over the ranges
for start, end, folder_name, new_start, output_name in ranges_2_1:
    if os.path.exists('new_temp'):
        shutil.rmtree('new_temp')
    os.makedirs("new_temp")
    for i in range(start, end + 1):
        src = f"{folder_name}/image_{i}.jpg"
        dst = f"new_temp/image_{new_start}.jpg"
        
        if os.path.exists(src):
            shutil.copy(src, dst)
            new_start += 1
    
    fd('src_temp', 'new_temp', output_name)

the box thresh:  1000
start to visualize the box on the images
drill2-1-1.GIF does not exist.
start to create GIF
the box thresh:  1000
start to visualize the box on the images
drill2-1-2.GIF does not exist.
start to create GIF
the box thresh:  1000
start to visualize the box on the images
drill2-1-3.GIF does not exist.
start to create GIF
the box thresh:  1000
start to visualize the box on the images
drill2-1-4.GIF does not exist.
start to create GIF
the box thresh:  1000
start to visualize the box on the images
drill2-1-5.GIF does not exist.
start to create GIF


benchmark GIF for first phase

In [21]:
create_gif_from_images('drill2-1-src.GIF', 'src_temp', '.jpg')

start to create GIF


phase 2

In [7]:
# Create folders
if os.path.exists('src_temp'):
    shutil.rmtree('src_temp')
os.makedirs("src_temp")

# Iterate over the ranges
for start, end, src_folder, new_start, _ in ranges_2_2_src:
    for i in range(start, end + 1):
        src = f"{src_folder}/image_{i}.jpg"
        dst = f"src_temp/image_{new_start}.jpg"
        
        if os.path.exists(src):
            shutil.copy(src, dst)
            new_start += 1

# Iterate over the ranges
for start, end, folder_name, new_start, output_name in ranges_2_2:
    if os.path.exists('new_temp'):
        shutil.rmtree('new_temp')
    os.makedirs("new_temp")
    for i in range(start, end + 1):
        src = f"{folder_name}/image_{i}.jpg"
        dst = f"new_temp/image_{new_start}.jpg"
        
        if os.path.exists(src):
            shutil.copy(src, dst)
            new_start += 1
    
    fd('src_temp', 'new_temp', output_name)

the box thresh:  1000
start to visualize the box on the images
drill2-2-1.GIF does not exist.
start to create GIF
the box thresh:  1000
start to visualize the box on the images
drill2-2-2.GIF does not exist.
start to create GIF


benchmark for second phase

In [24]:
create_gif_from_images('drill2-2-src.GIF', 'src_temp', '.jpg')

start to create GIF
